# 🎯 Clustering de Jugadores de FIFA (K-Means vs DBSCAN)

**Objetivo de esta clase práctica:**
1. Aplicar K-Means en datos reales con alta dimensionalidad (20+ columnas).
2. Aprender a **escalar** datos antes de usar distancia euclidiana.
3. Usar **PCA** únicamente para visualización (el clustering se hace en el espacio original).
4. Evaluar el número óptimo de clusters con **Silhouette Score** (dejando atrás el subjetivo "Método del Codo").
5. **Interpretar** los clusters calculando medias y poniéndoles nombres de verdad (ej: "Defensas", "Extremos").
6. Comparar con **DBSCAN** para ver cómo detecta outliers (jugadores "todoterreno" o rarezas).

---

## 1. Instalación de dependencias (si no las tienes)
Ejecuta esto UNA SOLA VEZ en tu terminal o en la primera celda si estás en Colab.

In [ ]:
!pip install kagglehub pandas numpy matplotlib seaborn scikit-learn

## 2. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples
import kagglehub
import os
import warnings
warnings.filterwarnings('ignore')

# Configuración visual para gráficos bonitos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Librerías importadas correctamente.")

## 3. Carga del Dataset de FIFA
Usaremos `kagglehub` para descargar automáticamente el dataset oficial de FIFA 22 (o 23) desde Kaggle.
*(Si falla la conexión, puedes comentar esta celda y cargar manualmente un CSV local).*

In [ ]:
# Descargar dataset de FIFA usando kagglehub
print("Descargando dataset de FIFA desde Kaggle...")
try:
    path = kagglehub.dataset_download("stefanoleone992/fifa-22-complete-player-dataset")
    file_path = os.path.join(path, "players_22.csv")  # O "players_20.csv" según el año
    df = pd.read_csv(file_path)
    print(f"✅ Dataset cargado correctamente. Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
except Exception as e:
    print("⚠️ No se pudo descargar automáticamente. Usando fallback a un CSV local.")
    print("Si tienes el archivo 'players_22.csv' en el mismo directorio, pásalo a df.")
    # Fallback: Si tienes el archivo localmente
    # df = pd.read_csv("players_22.csv")
    raise e

## 4. Exploración y Filtrado Inicial
Filtrar jugadores con **overall > 65** para eliminar a los jugadores muy bajos (menos relevantes) y mejorar la definición de los clusters.

In [ ]:
print("Vista previa de los datos:")
df.head()

In [ ]:
print(f"Columnas disponibles: {df.columns.tolist()[:10]} ... (y muchas más)")

**Filtro de calidad**: Solo jugadores con calificación general mayor a 65.

In [ ]:
df = df[df['overall'] > 65].copy()
print(f"Jugadores filtrados (overall > 65): {df.shape[0]}")

## 5. Selección de Features (Características numéricas relevantes)
Vamos a seleccionar las estadísticas de habilidades puras. 
Quitamos columnas de texto, identificadores (ID, nombre) y columnas financieras que distorsionan la distancia euclidiana (como `value_eur` o `wage_eur`) porque queremos clusterizar por **estilo de juego**, no por dinero.

In [ ]:
# %% [markdown]
# ## 5. Selección de Features y Limpieza de Datos (¡CORREGIDO!)

# %%
# Lista de features de habilidades (todas van del 0 al 99, excepto algunas)
features_habilidades = [
    'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physical',
    'attacking_crossing', 'attacking_finishing', 'attacking_heading_accuracy',
    'attacking_short_passing', 'attacking_volleys', 'skill_dribbling', 'skill_curve',
    'skill_fk_accuracy', 'skill_long_passing', 'skill_ball_control',
    'movement_acceleration', 'movement_sprint_speed', 'movement_agility',
    'movement_reactions', 'movement_balance', 'power_shot_power', 'power_jumping',
    'power_stamina', 'power_strength', 'power_long_shots', 'mentality_aggression',
    'mentality_interceptions', 'mentality_positioning', 'mentality_vision',
    'mentality_penalties', 'mentality_composure', 'defending_standing_tackle',
    'defending_sliding_tackle', 'goalkeeping_diving', 'goalkeeping_handling',
    'goalkeeping_kicking', 'goalkeeping_positioning', 'goalkeeping_reflexes'
]

# Verificamos cuáles de esas columnas existen realmente en nuestro dataset
features_existentes = [col for col in features_habilidades if col in df.columns]
print(f"✅ Features encontrados en el dataset: {len(features_existentes)} de {len(features_habilidades)}")

# Si faltan demasiados, usamos un subconjunto más básico y universal.
if len(features_existentes) < 15:
    print("⚠️ Usando subconjunto básico de features universales (6 columnas).")
    features_existentes = ['pace', 'shooting', 'passing', 'dribbling', 'defending', 'physical']

# 🔥 CORRECCIÓN CLAVE: Eliminar filas que tengan algún valor faltante (NaN) en estas columnas.
print(f"📊 Filas antes de limpiar NaNs: {df.shape[0]}")
df = df.dropna(subset=features_existentes)
print(f"📊 Filas después de limpiar NaNs: {df.shape[0]}")

# Extraer la matriz de características
X = df[features_existentes].values

# Verificar que no haya quedado vacío
if X.shape[0] == 0:
    raise ValueError("❌ No quedaron datos después de eliminar NaNs. Revisa el filtro 'overall > 65' o las columnas seleccionadas.")

print(f"✅ Matriz de características creada con éxito: {X.shape[0]} filas y {X.shape[1]} columnas.")

## 6. ¡CRUCIAL! Estandarización de los datos (StandardScaler)
**¿Por qué es obligatorio?** 
- Las habilidades van de 0 a 99. 
- Pero si incluyéramos `value_eur` (millones de euros), esa columna dominaría la distancia euclidiana.
- Incluso entre habilidades, la escala es similar (0-99), pero es buena práctica estandarizar para darle el mismo peso a todas. 
- La distancia euclidiana se calcula en el espacio transformado.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Datos estandarizados: Media ~0, Desviación ~1 en cada feature.")
print(f"Ejemplo de primeras 5 filas escaladas:\n{X_scaled[:5, :4]}")

## 7. Reducción de Dimensionalidad con PCA (SOLO PARA VISUALIZACIÓN)
**Importante**: Vamos a reducir a 2D *únicamente* para pintar el gráfico. 
El algoritmo K-Means se entrenará con los datos **originales escalados (X_scaled)** de 20+ dimensiones.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Varianza explicada por Componente 1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"Varianza explicada por Componente 2: {pca.explained_variance_ratio_[1]:.2%}")

## 8. Encontrar el número óptimo de Clusters (Silhouette Score)
En lugar del subjetivo "Método del Codo", usamos el **Coeficiente de Silueta**.
El valor más alto de Silhouette = mejor número de clusters (porque los grupos están más separados y compactos).

In [ ]:
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k} -> Silhouette Score: {score:.4f}")

# Gráfico de los scores
plt.figure(figsize=(10, 5))
plt.plot(K_range, silhouette_scores, marker='o', linestyle='--', color='b')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score para diferentes K (FIFA)')
plt.grid(True)
plt.xticks(K_range)
plt.show()

**Interpretación:** Generalmente K=4 o K=5 son los mejores. En fútbol, los roles tácticos principales son 4: Portero, Defensa, Mediocampista, Delantero. Usaremos K=4.

## 9. Entrenar K-Means (K=4)


In [ ]:
k_optimo = 4
kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
df['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

# Guardamos los centroides en el espacio original escalado para futuras predicciones
centroides = scaler.inverse_transform(kmeans.cluster_centers_)  # Opcional: volver a escala original

print("✅ Clusters asignados a cada jugador.")

## 10. 🧠 INTERPRETACIÓN PROFESIONAL (El momento estrella)
Vamos a agrupar por cluster y calcular la **media** de las habilidades originales (sin escalar) para ver qué representa cada grupo.

In [ ]:
# Función para bautizar clusters
def bautizar_cluster(fila):
    if fila['goalkeeping_diving'] > 60:  # Si tiene atributos de portero altos
        return "🧤 Porteros"
    elif fila['defending'] > 70 and fila['physical'] > 70:
        return "🛡️ Defensas / Contención"
    elif fila['pace'] > 80 and fila['dribbling'] > 75:
        return "⚡ Extremos / Wingers"
    elif fila['shooting'] > 75 and fila['passing'] > 70:
        return "🎯 Delanteros / Mediapuntas"
    elif fila['passing'] > 75 and fila['dribbling'] > 70:
        return "🧠 Mediocampistas Creativos"
    else:
        return "🔄 Todoterreno / Mixto"

## 11. Visualización en 2D (PCA) coloreada por K-Means

In [ ]:
plt.figure(figsize=(14, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], 
                       c=df['cluster_kmeans'], 
                       cmap='viridis', 
                       alpha=0.6, 
                       s=10)

plt.title(f'Segmentación de Jugadores FIFA (K-Means, K={k_optimo}) en plano PCA')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.colorbar(scatter, label='Cluster')
plt.grid(True)

# Dibujamos los centroides de los clusters en el espacio PCA (proyectados)
centroides_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centroides_pca[:, 0], centroides_pca[:, 1], 
            c='red', marker='X', s=200, label='Centroides (PCA)')
plt.legend()
plt.show()

## 12. Desafío: DBSCAN aplicado a los mismos datos
DBSCAN no necesita que le digamos K. Agrupa por *densidad* y marca como ruido (-1) a los puntos atípicos.

**Ajuste de parámetros:**
- `eps`: Distancia máxima para considerar a otro punto como vecino. (En datos escalados, solemos probar entre 0.3 y 0.8).
- `min_samples`: Número mínimo de vecinos para formar un cluster denso.

In [ ]:
# Probamos DBSCAN con un eps de 0.5 (porque los datos están estandarizados)
dbscan = DBSCAN(eps=0.6, min_samples=10)
df['cluster_dbscan'] = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(df['cluster_dbscan'])) - (1 if -1 in df['cluster_dbscan'] else 0)
n_outliers = sum(df['cluster_dbscan'] == -1)

print(f"📌 DBSCAN encontró {n_clusters_db} clusters densos.")
print(f"⚠️ Jugadores considerados como RUIDO (outliers) : {n_outliers} ({n_outliers/len(df)*100:.2f}%)")

**Visualización de DBSCAN vs K-Means:** 
Vemos cómo DBSCAN deja en gris a los jugadores que no encajan perfectamente en un rol puro (ej: mediocampistas box-to-box, o jugadores con habilidades muy equilibradas).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot K-Means
ax1 = axes[0]
ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster_kmeans'], cmap='viridis', s=10, alpha=0.6)
ax1.set_title('K-Means (K=4)')
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')

# Plot DBSCAN
ax2 = axes[1]
scatter2 = ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster_dbscan'], cmap='tab10', s=10, alpha=0.6)
ax2.set_title(f'DBSCAN (Clusters={n_clusters_db}, Ruido={n_outliers})')
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
plt.colorbar(scatter2, ax=ax2)

plt.tight_layout()
plt.show()

## 13. Análisis de los Outliers de DBSCAN
Veamos qué características tienen esos jugadores marcados como ruido. Suele ser que tienen habilidades muy distribuidas (todoterreno).

In [ ]:
# %% [markdown]
# ## 13. Análisis de los Outliers de DBSCAN (CORREGIDO)
# Veamos qué características tienen esos jugadores marcados como ruido. Suele ser que tienen habilidades muy distribuidas (todoterreno).

# %%
outliers_df = df[df['cluster_dbscan'] == -1]
print(f"👀 Ejemplo de 5 jugadores considerados ruido (outliers):")

if len(outliers_df) > 0:
    # 🔥 Lista de columnas que queremos mostrar (con nombres que pueden variar)
    columnas_deseadas = ['short_name', 'overall', 'pace', 'shooting', 'passing', 'defending', 'physical', 'physic']
    
    # Filtramos SOLO las que realmente existen en el DataFrame
    columnas_existentes = [col for col in columnas_deseadas if col in df.columns]
    
    if not columnas_existentes:
        print("⚠️ Ninguna de las columnas esperadas ('short_name', 'overall', etc.) existe. Mostrando las primeras 5 columnas disponibles:")
        display(outliers_df.iloc[:, :5].head())
    else:
        print(f"🔍 Mostrando columnas: {columnas_existentes}")
        display(outliers_df[columnas_existentes].head())
else:
    print("No se encontraron outliers con estos parámetros.")